In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Main libraries
import numpy as np
import os

# Plotting
import matplotlib.pyplot as plt
import seaborn as sns

# We will use Polars for data manipulation
import polars as pl

# Casting types from time to time to have a better autocompletion

from models import train_and_explain, ExperimentResults, Species, ModelType, ALL_SPECIES
from analysis import summarize_performance
from config import Ablation, FEATURES_METADATA
from scipy.stats import lognorm
import cmocean

CMAP = cmocean.cm.thermal

# Use caching for various results
if not os.path.exists("./cache"):
    os.makedirs("./cache")

In [ ]:
# Pick grouping strategy: "tree_id" or "plot_id" or None
group_col = "tree_id"
# Pick model type: "gbdt" or "lasso"
model_type: ModelType = "gbdt"
# Ablation: "all", "tree-level-only", "plot-level-only", "no-defoliation", "max-defoliation"
ablation: Ablation = "all"
use_temporal_cv = True

In [ ]:
# Train and explain models for all species
# Set use_temporal_cv = True, to enable HierarchicalTemporalGroup
all_results: dict[Species, ExperimentResults] = {}

for species in ALL_SPECIES:
    all_results[species] = train_and_explain(
        species,
        model_type=model_type,
        group_by=group_col,
        ablation=ablation,
        use_temporal_cv=use_temporal_cv,
    )

In [ ]:
# Summarize performance as a table
summarize_performance(
    all_results, ablation=ablation, model_type=model_type, group_col=group_col
)

In [ ]:
from sklearn.metrics import r2_score

r2_results = []


def inv_transform(y_quantile, shape, loc, scale):
    y_quantile = np.clip(y_quantile, 1e-10, 1 - 1e-10)
    return lognorm.ppf(y_quantile, shape, loc, scale) - 1


for species, results in all_results.items():
    shape, loc, scale = results.dist_params

    for fold in range(results.num_folds):
        X_train, y_true_train, y_pred_train = results.get_data(fold, split="train")
        X_test, y_true_test, y_pred_test = results.get_data(fold, split="test")

        # Transform to original units for train
        y_true_orig_train = inv_transform(
            np.clip(y_true_train.to_numpy(), 1e-10, 1 - 1e-10), shape, loc, scale
        )
        y_pred_orig_train = inv_transform(
            np.clip(y_pred_train.to_numpy(), 1e-10, 1 - 1e-10), shape, loc, scale
        )

        # Transform to original units for test
        y_true_orig_test = inv_transform(
            np.clip(y_true_test.to_numpy(), 1e-10, 1 - 1e-10), shape, loc, scale
        )
        y_pred_orig_test = inv_transform(
            np.clip(y_pred_test.to_numpy(), 1e-10, 1 - 1e-10), shape, loc, scale
        )

        r2_results.append(
            {
                "species": species,
                "fold": fold,
                "n_train": len(X_train),
                "n_test": len(X_test),
                "quantile_r2 (train)": r2_score(y_true_train, y_pred_train),
                "quantile_r2 (test)": r2_score(y_true_test, y_pred_test),
                "growth_r2 (train)": r2_score(y_true_orig_train, y_pred_orig_train),
                "growth_r2 (test)": r2_score(y_true_orig_test, y_pred_orig_test),
            }
        )

df_r2 = pl.DataFrame(r2_results)

In [ ]:
summary = df_r2.group_by("species").agg(
    [
        # Quantile R²
        pl.col("quantile_r2 (test)").mean().alias("quantile_r2_mean"),
        pl.col("quantile_r2 (test)").std().alias("quantile_r2_std"),
        (
            pl.col("quantile_r2 (test)").std()
            / np.sqrt(pl.col("quantile_r2 (test)").count())
        ).alias("quantile_r2_se"),
        # Growth R²
        pl.col("growth_r2 (test)").mean().alias("growth_r2_mean"),
        pl.col("growth_r2 (test)").std().alias("growth_r2_std"),
        (
            pl.col("growth_r2 (test)").std()
            / np.sqrt(pl.col("growth_r2 (test)").count())
        ).alias("growth_r2_se"),
    ]
)

print(
    f"{'Species':<10} | {'Quantile R² (reported)':<22} | {'Growth R² (original units)':<25}|"
)
print("-" * 65)
for row in summary.iter_rows():
    species = row[0]
    q_mean, q_se = row[1], row[3]
    g_mean, g_se = row[4], row[6]
    quantile_str = f"{q_mean:.3f} ± {q_se:.3f}"
    growth_str = f"{g_mean:.3f} ± {g_se:.3f}"
    print(f"{species:<10} | {quantile_str:<22} | {growth_str:<25} |")

In [ ]:
rows = []

for species, results in all_results.items():
    if results.dist_params is None:
        raise ValueError(f"{species}: dist_params is None")

    shape, loc, scale = results.dist_params

    for fold in range(results.num_folds):
        X_train, y_true_train, y_pred_train = results.get_data(fold, split="train")
        X_test, y_true_test, y_pred_test = results.get_data(fold, split="test")
        y_pred = np.concatenate(
            [
                y_pred_train.to_numpy(),
                y_pred_test.to_numpy(),
            ]
        )
        feature_names = list(X_train.columns)
        shap_values = results.shap_values[fold].values
        base_values = results.shap_values[fold].base_values
        u_pred = base_values + shap_values.sum(axis=1)

        # Check if SHAP reconstructs predictions
        if not np.allclose(u_pred, y_pred, rtol=1e-6, atol=1e-6):
            max_abs_err = np.max(np.abs(u_pred - y_pred))
            mean_abs_err = np.mean(np.abs(u_pred - y_pred))

            print(
                f"[WARNING] {species} fold {fold}: "
                f"u_pred != y_pred "
                f"(max abs err={max_abs_err:.6g}, "
                f"mean abs err={mean_abs_err:.6g})"
            )

        y_pred_orig = inv_transform(u_pred, shape, loc, scale)
        u_without = u_pred[:, None] - shap_values
        y_without = inv_transform(u_without, shape, loc, scale)
        delta_abs = np.abs(y_pred_orig[:, None] - y_without)
        rows.extend(
            {
                "species": species,
                "fold": fold,
                "feature": feature_names[j],
                "shap": float(delta_abs[i, j]),
            }
            for i in range(len(y_pred))
            for j in range(len(X_train.columns))
        )

feature_importances = pl.DataFrame(rows)

feature_importances = feature_importances.group_by(
    ["species", "fold", "feature"]
).mean()

In [ ]:
min_importance = 1.0
max_rank = 3

data = (
    feature_importances.with_columns(
        importance=pl.col("shap").mean().over("species", "feature") * 100
    )
    .with_columns(
        rank=pl.col("importance").rank(descending=True, method="dense").over("species"),
    )
    .filter(
        (
            (pl.col("importance").max().over("feature") >= min_importance)
            | (pl.col("rank").min().over("feature") <= max_rank)
        )
    )
    .join(
        pl.from_dicts(
            [
                {"feature": k, "feature_label": v["label"]}
                for k, v in FEATURES_METADATA.items()
            ]
        ),
        on="feature",
        how="left",
    )
    .with_columns(feature=pl.col("feature_label"))
    .drop("feature_label")
    .sort(
        "species", pl.col("importance").mean().over("feature"), descending=[False, True]
    )
)
# Add a summary over all species
data = pl.concat(
    [
        data,
        data.group_by("feature", "fold")
        .agg(
            shap=pl.col("shap").mean(),
            importance=pl.col("importance").mean(),
        )
        .with_columns(species=pl.lit("all species"))
        .with_columns(
            rank=pl.col("importance")
            .rank(descending=True, method="dense")
            .over("species"),
        )
        .select(data.columns),
    ],
    how="vertical_relaxed",
)


with pl.Config() as cfg:
    cfg.set_tbl_formatting("ASCII_FULL")
    cfg.set_tbl_rows(100)

    stats = (
        data.filter(pl.col("species") != "all species")
        .group_by("feature", "species")
        .agg(pl.mean("rank"))
        .sort(by="rank")
        .filter(pl.col("feature") == "Mean defoliation")
    )

    print(stats)

data

In [ ]:
# Plot using seaborn
num_species = len(feature_importances.select("species").unique())
g = sns.catplot(
    data,
    x="shap",
    y="feature",
    hue="species",
    kind="bar",
    palette=sns.color_palette("cmo.thermal", n_colors=num_species + 1),
    height=8,
    aspect=0.6,
)

g._legend.set_title("Species")

# Capitalize legend labels
new_labels = [label.get_text().capitalize() for label in g._legend.texts[0:]]
for label, new_text in zip(g._legend.texts[0:], new_labels):
    label.set_text(new_text)

# Prepend rank to each feature name of the y-axis
ax = g.facet_axis(0, 0)
yticks = ax.get_yticks()
ylabels = ax.get_yticklabels()
new_ylabels = []
for ytick, ylabel in zip(yticks, ylabels):
    feature_name = ylabel.get_text()
    rank = int(
        data.group_by("feature", "species")
        .agg(pl.col("rank").mean())
        .filter(
            (pl.col("feature") == feature_name) & (pl.col("species") == "all species")
        )
        .select("rank")
        .item()
    )
    new_ylabels.append(f"({rank}) {feature_name}")
ax.set_yticklabels(new_ylabels)

plt.xlabel("Feature importance (mean |SHAP value| %)")
plt.ylabel("Feature")

model_label = "GBDT" if model_type == "gbdt" else "Lasso"
group_label = "tree identifier" if group_col == "tree_id" else "plot identifier"
ablation_label = "all features" if ablation == "all" else "without defoliation"
plt.title(f"Feature importance ({model_label}, {ablation_label})")

fig = plt.gcf()
plt.savefig(
    f"./figures/org-space-importance-{model_type}-{group_col}-{ablation}.pdf",
    bbox_inches="tight",
)